# LeetCode 1974 — Minimum Time to Type Word Using Special Typewriter bằng Python

**Link đề:** https://leetcode.com/problems/minimum-time-to-type-word-using-special-typewriter/
**Độ khó:** Easy · **Chủ đề:** String, Greedy

```python
class Solution:
    def minTimeToType(self, word: str) -> int:
        ...
```

## Mục tiêu học
1. Mô hình hoá 26 chữ cái xếp **thành vòng tròn** bằng chỉ số `0..25` và phép **modulo** `% 26`.
2. Tính **khoảng cách ngắn nhất trên vòng tròn**: `min(d, 26 - d)` với `d = |a - b|`.
3. Đi từ mô phỏng từng bước di chuyển (baseline) → công thức `O(1)` mỗi ký tự.
4. Hiểu vì sao **tham lam** (mỗi lần chọn chiều ngắn hơn) là tối ưu toàn cục.
5. Trace, chứng minh, Time/Space, lỗi hay gặp, biến thể (bàn phím vòng tròn kích thước bất kỳ, khoá số…).

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — máy đánh chữ vòng tròn

Máy đánh chữ đặc biệt có 26 chữ `'a'…'z'` xếp **thành vòng tròn**, với một **con trỏ** ban đầu chỉ vào `'a'`. Chỉ gõ được chữ mà con trỏ đang chỉ.

Mỗi giây, ta làm **một** trong hai thao tác:
- dịch con trỏ **một** ô theo chiều kim đồng hồ hoặc ngược chiều;
- **gõ** chữ con trỏ đang chỉ.

Cho `word`, trả về **số giây ít nhất** để gõ xong `word`.

| `word` | Output | Giải thích |
|---|:---:|---|
| `"abc"` | `5` | gõ a (1) · sang b (1) · gõ (1) · sang c (1) · gõ (1) |
| `"bza"` | `7` | sang b (1) · gõ (1) · **ngược** về z (2) · gõ (1) · sang a (1) · gõ (1) |
| `"zjpc"` | `34` | ngược về z (1) · gõ · tới j (10) · gõ · tới p (6) · gõ · ngược về c (13) · gõ |

**Constraints:** `1 <= word.length <= 100`, chỉ chữ thường.

Nhận xét ngay: mỗi chữ tốn đúng **1 giây để gõ** → phần gõ là `len(word)` giây; phần còn lại là tổng số bước di chuyển.

In [1]:
# Ví dụ "zjpc": tách thời gian di chuyển và thời gian gõ
moves = [1, 10, 6, 13]
print("di chuyển:", sum(moves), "| gõ:", len("zjpc"), "| tổng:", sum(moves) + len("zjpc"))   # 30 + 4 = 34

di chuyển: 30 | gõ: 4 | tổng: 34


## 2. Nền tảng Python và toán cần dùng

### 2.1. Đổi chữ thành vị trí `0..25`

`ord(ch) - ord('a')` cho `a → 0`, `b → 1`, …, `z → 25`. Dùng chỉ số bắt đầu từ 0 giúp công thức modulo gọn.

In [2]:
for ch in "abjpz":
    print(ch, ord(ch) - ord("a"))

a 0
b 1
j 9
p 15
z 25


### 2.2. `abs` — giá trị tuyệt đối

`abs(x)` bỏ dấu âm. Khoảng cách "đi thẳng" (không vòng qua `z → a`) giữa vị trí `p` và `q` là `abs(p - q)`.

In [3]:
print(abs(3 - 10), abs(10 - 3), abs(0 - 25))   # 7 7 25

7 7 25


### 2.3. Vòng tròn và phép `% 26`

Trên vòng tròn 26 ô, đi theo chiều kim đồng hồ một bước từ `z` (25) là tới `a` (0): `(25 + 1) % 26 = 0`. Đi ngược một bước từ `a`: `(0 - 1) % 26 = 25`.

Trong Python, `%` với số dương ở vế phải **luôn trả kết quả không âm** (khác C/Java), rất tiện cho vòng tròn.

In [4]:
print((25 + 1) % 26, (0 - 1) % 26, (-27) % 26)   # 0 25 25
print(-1 % 26, "khác với C/Java (ở đó -1 % 26 == -1)")

0 25 25
25 khác với C/Java (ở đó -1 % 26 == -1)


### 2.4. Hai chiều đi trên vòng tròn

Từ `p` tới `q`:
- **chiều kim đồng hồ** (tăng chỉ số): `(q - p) % 26` bước;
- **ngược chiều** (giảm chỉ số): `(p - q) % 26` bước.

Hai số này cộng lại bằng `26` (trừ khi `p == q`, lúc đó cả hai là `0`). Ta chọn số **nhỏ hơn**.

In [5]:
p, q = ord("b") - 97, ord("z") - 97
cw, ccw = (q - p) % 26, (p - q) % 26
print(f"b -> z: theo chiều kim đồng hồ {cw} bước, ngược chiều {ccw} bước, tổng {cw + ccw}")

b -> z: theo chiều kim đồng hồ 24 bước, ngược chiều 2 bước, tổng 26


### 2.5. `zip` để duyệt từng cặp liền kề

`zip("a" + word, word)` cho các cặp `(chữ trước, chữ hiện tại)`, với chữ trước của chữ đầu tiên là `'a'` (vị trí con trỏ ban đầu).

In [6]:
word = "bza"
for prev, cur in zip("a" + word, word):
    print(prev, "->", cur)

a -> b
b -> z
z -> a


## 3. Tính tay — ví dụ `"zjpc"`

| Bước | Từ → tới | vị trí | `d = |p - q|` | `26 - d` | chọn `min` | + gõ |
|---:|---|---|---:|---:|---:|---:|
| 1 | a → z | 0 → 25 | 25 | 1 | 1 | 1 |
| 2 | z → j | 25 → 9 | 16 | 10 | 10 | 1 |
| 3 | j → p | 9 → 15 | 6 | 20 | 6 | 1 |
| 4 | p → c | 15 → 2 | 13 | 13 | 13 | 1 |

Tổng di chuyển `1 + 10 + 6 + 13 = 30`, gõ `4` → `34`. Bước 4 là ca "**ngang nhau**" (13 = 13): chọn chiều nào cũng được.

## 4. Cách 1 — Baseline: mô phỏng từng bước dịch con trỏ

Ý tưởng trực giác: với mỗi chữ cần gõ, **thử đi** theo chiều kim đồng hồ từng bước cho tới khi gặp chữ đó (đếm bước), rồi thử chiều ngược lại; lấy chiều ít bước hơn. Đây chính là cách "người thật" làm với máy.

In [7]:
def steps_by_walking(start, target, direction):
    # direction = +1: chiều kim đồng hồ, -1: ngược chiều
    pos, steps = start, 0
    while pos != target:
        pos = (pos + direction) % 26
        steps += 1
    return steps

def min_time_simulate(word):
    pointer = 0
    total = 0
    for ch in word:
        target = ord(ch) - ord("a")
        cw = steps_by_walking(pointer, target, +1)
        ccw = steps_by_walking(pointer, target, -1)
        total += min(cw, ccw) + 1          # di chuyển + 1 giây gõ
        pointer = target
    return total

print(min_time_simulate("abc"), min_time_simulate("bza"), min_time_simulate("zjpc"))   # 5 7 34

5 7 34


### 4.1. Chi phí của mô phỏng

Mỗi chữ có thể tốn tới `25 + 25` bước đi thử → `O(26 · n)`. Với `n ≤ 100` hoàn toàn ổn, nhưng nếu vòng tròn có `K = 10^9` ô (biến thể) thì mô phỏng từng bước là không thể. Ta cần **công thức**.

In [8]:
walk_steps = sum(steps_by_walking(0, t, +1) + steps_by_walking(0, t, -1) for t in range(26))
print("tổng số bước thử khi đi từ 'a' tới mọi chữ:", walk_steps)   # 26 * 26 - 26 = 650 (mỗi t != 0 tốn đúng 26)

tổng số bước thử khi đi từ 'a' tới mọi chữ: 650


## 5. Bước đột phá — khoảng cách vòng tròn `min(d, 26 - d)`

Từ mục 2.4: hai chiều tốn `cw` và `ccw` bước với `cw + ccw = 26` (khi `p ≠ q`). Mà `d = |p - q|` chính là một trong hai số đó (chiều "không vượt qua mốc z|a"), số còn lại là `26 - d`. Vậy:

```text
moves(p, q) = min(d, 26 - d),   d = |p - q|
```

Ca `p == q`: `d = 0`, `min(0, 26) = 0` — công thức vẫn đúng, không cần xử lý riêng.

In [9]:
# Kiểm chứng công thức với mô phỏng cho mọi cặp (p, q)
for p in range(26):
    for q in range(26):
        d = abs(p - q)
        assert min(d, 26 - d) == min(steps_by_walking(p, q, +1), steps_by_walking(p, q, -1))
print("Công thức min(d, 26 - d) khớp mô phỏng trên cả 26 x 26 = 676 cặp")

Công thức min(d, 26 - d) khớp mô phỏng trên cả 26 x 26 = 676 cặp


## 6. Lời giải chính — giảng từng đoạn

**Đoạn 1:** `pointer = 0` (đang chỉ `'a'`), `total = 0`.

**Đoạn 2 — mỗi `ch`:** `target = ord(ch) - ord('a')`; `d = abs(target - pointer)`; cộng `min(d, 26 - d)` (di chuyển) và `1` (gõ).

**Đoạn 3:** cập nhật `pointer = target` — con trỏ **ở lại** chữ vừa gõ.

**Đoạn 4:** trả `total`.

In [10]:
def min_time_to_type(word):
    pointer = 0                                   # Đoạn 1
    total = 0
    for ch in word:                               # Đoạn 2
        target = ord(ch) - ord("a")
        d = abs(target - pointer)
        total += min(d, 26 - d) + 1
        pointer = target                          # Đoạn 3
    return total                                  # Đoạn 4

print(min_time_to_type("abc"), min_time_to_type("bza"), min_time_to_type("zjpc"))   # 5 7 34

5 7 34


## 7. TRACE từng bước

Ví dụ `word = "bza"`:

| ch | pointer trước | target | d | 26 − d | di chuyển | + gõ | total sau |
|---|---:|---:|---:|---:|---:|---:|---:|
| b | 0 | 1 | 1 | 25 | 1 | 1 | 2 |
| z | 1 | 25 | 24 | **2** | 2 | 1 | 5 |
| a | 25 | 0 | 25 | **1** | 1 | 1 | 7 |

Hai bước sau đều chọn chiều **vòng qua** mốc `z|a` vì `26 − d` nhỏ hơn.

In [11]:
def trace_min_time(word):
    pointer, total = 0, 0
    print(f"word = {word!r}")
    for ch in word:
        target = ord(ch) - ord("a")
        d = abs(target - pointer)
        move = min(d, 26 - d)
        how = "đi thẳng" if d <= 26 - d else "vòng qua z|a"
        total += move + 1
        print(f"  {chr(pointer + 97)}({pointer:>2}) -> {ch}({target:>2}): d={d:>2}, 26-d={26 - d:>2}, "
              f"di chuyển {move:>2} ({how}), +1 gõ, total = {total}")
        pointer = target
    return total

print("Kết quả:", trace_min_time("bza"))
print()
print("Kết quả:", trace_min_time("zjpc"))

word = 'bza'
  a( 0) -> b( 1): d= 1, 26-d=25, di chuyển  1 (đi thẳng), +1 gõ, total = 2
  b( 1) -> z(25): d=24, 26-d= 2, di chuyển  2 (vòng qua z|a), +1 gõ, total = 5
  z(25) -> a( 0): d=25, 26-d= 1, di chuyển  1 (vòng qua z|a), +1 gõ, total = 7
Kết quả: 7

word = 'zjpc'
  a( 0) -> z(25): d=25, 26-d= 1, di chuyển  1 (vòng qua z|a), +1 gõ, total = 2
  z(25) -> j( 9): d=16, 26-d=10, di chuyển 10 (vòng qua z|a), +1 gõ, total = 13
  j( 9) -> p(15): d= 6, 26-d=20, di chuyển  6 (đi thẳng), +1 gõ, total = 20
  p(15) -> c( 2): d=13, 26-d=13, di chuyển 13 (đi thẳng), +1 gõ, total = 34
Kết quả: 34


`chr(pointer + 97)` đổi vị trí ngược lại thành chữ (vì `ord('a') == 97`).

### 7.1. Nhánh đặc biệt: gõ lại cùng một chữ và ca ngang nhau

In [12]:
print("Kết quả:", trace_min_time("aan"))     # 'a' -> 'a': 0 bước; 'a' -> 'n': d = 13 = 26 - 13

word = 'aan'
  a( 0) -> a( 0): d= 0, 26-d=26, di chuyển  0 (đi thẳng), +1 gõ, total = 1
  a( 0) -> a( 0): d= 0, 26-d=26, di chuyển  0 (đi thẳng), +1 gõ, total = 2
  a( 0) -> n(13): d=13, 26-d=13, di chuyển 13 (đi thẳng), +1 gõ, total = 16
Kết quả: 16


## 8. Các cách viết khác

### 8.1. Cách 3 — dùng modulo cho hai chiều

In [13]:
def min_time_mod(word):
    pointer, total = 0, 0
    for ch in word:
        target = ord(ch) - ord("a")
        cw = (target - pointer) % 26
        ccw = (pointer - target) % 26
        total += min(cw, ccw) + 1
        pointer = target
    return total

print(min_time_mod("abc"), min_time_mod("bza"), min_time_mod("zjpc"))

5 7 34


### 8.2. Cách 4 — tách "gõ" và "di chuyển", dùng `zip` + `sum`

Thời gian = `len(word)` (gõ) + tổng khoảng cách vòng tròn giữa các cặp liền kề, bắt đầu từ `'a'`.

In [14]:
def circular_distance(x, y, size=26):
    d = abs(ord(x) - ord(y))
    return min(d, size - d)

def min_time_zip(word):
    return len(word) + sum(circular_distance(p, c) for p, c in zip("a" + word, word))

print(min_time_zip("abc"), min_time_zip("bza"), min_time_zip("zjpc"))

5 7 34


### 8.3. Cách 5 — bảng tra khoảng cách tính sẵn

Chỉ có `26 × 26` cặp; tính trước một bảng 2 chiều (list lồng list) rồi tra. Hữu ích khi gọi hàm rất nhiều lần.

In [15]:
DIST = [[min(abs(p - q), 26 - abs(p - q)) for q in range(26)] for p in range(26)]
print(DIST[0][25], DIST[25][9], DIST[15][2])   # 1 10 13

def min_time_table(word):
    pointer, total = 0, 0
    for ch in word:
        t = ord(ch) - 97
        total += DIST[pointer][t] + 1
        pointer = t
    return total

print(min_time_table("zjpc"))

1 10 13
34


## 9. Code nộp LeetCode (Python 3)

In [16]:
class Solution:
    def minTimeToType(self, word: str) -> int:
        pointer = 0
        total = 0
        for ch in word:
            target = ord(ch) - ord("a")
            d = abs(target - pointer)
            total += min(d, 26 - d) + 1
            pointer = target
        return total


sol = Solution()
assert sol.minTimeToType("abc") == 5
assert sol.minTimeToType("bza") == 7
assert sol.minTimeToType("zjpc") == 34
print("3 ví dụ của đề PASS")

3 ví dụ của đề PASS


## 10. Vì sao thuật toán đúng (tham lam là tối ưu)?

**1. Cấu trúc lời giải bất kỳ.** Ta phải gõ `word[0], word[1], …` **theo thứ tự**. Giữa lần gõ `word[i-1]` và lần gõ `word[i]`, con trỏ phải đi từ vị trí `word[i-1]` tới vị trí `word[i]` (với `word[-1]` hiểu là `'a'`). Mọi lời giải hợp lệ đều chia thành `n` đoạn như vậy, cộng `n` giây gõ.

**2. Các đoạn độc lập.** Điểm đầu và điểm cuối của mỗi đoạn **cố định** (do chữ cần gõ quyết định), không phụ thuộc cách đi ở đoạn khác. Vì vậy tổng nhỏ nhất = tổng của các **giá trị nhỏ nhất từng đoạn** — đây là lý do tham lam đúng ở bài này.

**3. Mỗi đoạn tối thiểu là `min(d, 26 - d)`.** Mỗi giây con trỏ dịch ±1 trên vòng tròn. Đi một số bước theo cả hai chiều xen kẽ chỉ làm "bù trừ" và tốn thêm. Đi chỉ một chiều: chiều kim đồng hồ cần `(q - p) mod 26` bước, chiều ngược cần `(p - q) mod 26` bước; hai giá trị này là `d` và `26 - d` (theo thứ tự nào đó). Không thể ít hơn số nhỏ hơn trong hai.

**Invariant vòng lặp:** sau khi xử lý `i` chữ đầu, `total` = thời gian tối thiểu để gõ `word[:i]` và `pointer` = vị trí `word[i-1]`.

## 11. Time / Space Complexity

Gọi `n = len(word)`, `K = 26` (số ô trên vòng tròn).

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| 1. Mô phỏng từng bước | `O(n · K)` | `O(1)` | trực giác, không mở rộng được với `K` lớn |
| 2. `min(d, K - d)` (chính) | `O(n)` | `O(1)` | nên nộp |
| 3. Modulo hai chiều | `O(n)` | `O(1)` | tương đương |
| 4. `zip` + `sum` | `O(n)` | `O(n)` (chuỗi `"a" + word`) | gọn |
| 5. Bảng tra | `O(K² + n)` | `O(K²)` | lợi khi gọi rất nhiều lần |

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.minTimeToType("abc"), 5)
check("đề 2", sol.minTimeToType("bza"), 7)
check("đề 3", sol.minTimeToType("zjpc"), 34)
check("chỉ 'a'", sol.minTimeToType("a"), 1)
check("lặp cùng chữ", sol.minTimeToType("aaaa"), 4)
check("a -> z vòng qua", sol.minTimeToType("z"), 2)
check("ca ngang nhau a -> n", sol.minTimeToType("n"), 14)
check("a -> m (đi thẳng 12)", sol.minTimeToType("m"), 13)
check("a -> o (vòng 12)", sol.minTimeToType("o"), 13)
check("qua lại z, a", sol.minTimeToType("zaza"), 8)

PASS đề 1: 5
PASS đề 2: 7
PASS đề 3: 34
PASS chỉ 'a': 1
PASS lặp cùng chữ: 4
PASS a -> z vòng qua: 2
PASS ca ngang nhau a -> n: 14
PASS a -> m (đi thẳng 12): 13
PASS a -> o (vòng 12): 13
PASS qua lại z, a: 8


In [18]:
# Đối chiếu các cách trên dữ liệu ngẫu nhiên có seed
import random
random.seed(1974)
funcs = [min_time_simulate, min_time_to_type, min_time_mod, min_time_zip, min_time_table, Solution().minTimeToType]
for _ in range(500):
    w = "".join(random.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(random.randint(1, 100)))
    results = {f(w) for f in funcs}
    assert len(results) == 1, (w, results)
print("500 từ ngẫu nhiên: mọi cách khớp nhau")
# Cận trên: mỗi chữ tối đa 13 bước + 1 gõ
print("cận trên với n = 100:", 100 * 14, "| ví dụ 'an' * 50 cho:", sol.minTimeToType("an" * 50))

500 từ ngẫu nhiên: mọi cách khớp nhau
cận trên với n = 100: 1400 | ví dụ 'an' * 50 cho: 1387


## 13. Sai lầm người mới hay mắc

### Lỗi 1 — Quên chiều vòng qua (chỉ dùng `abs`)

In [19]:
def buggy_abs_only(word):
    pointer, total = 0, 0
    for ch in word:
        t = ord(ch) - 97
        total += abs(t - pointer) + 1      # chỉ đi thẳng
        pointer = t
    return total

print("buggy:", buggy_abs_only("bza"), "| đúng:", min_time_to_type("bza"))   # 53 vs 7

buggy: 53 | đúng: 7


### Lỗi 2 — Quên cộng 1 giây gõ, hoặc cộng `len(word)` hai lần

In [20]:
def buggy_no_typing(word):
    pointer, total = 0, 0
    for ch in word:
        t = ord(ch) - 97
        d = abs(t - pointer)
        total += min(d, 26 - d)            # quên + 1
        pointer = t
    return total

print("buggy:", buggy_no_typing("abc"), "| đúng:", min_time_to_type("abc"))   # 2 vs 5

buggy: 2 | đúng: 5


### Lỗi 3 — Không cập nhật `pointer` (luôn đo từ `'a'`)

In [21]:
def buggy_no_update(word):
    total = 0
    for ch in word:
        d = ord(ch) - 97                   # luôn so với 'a'
        total += min(d, 26 - d) + 1
    return total

print("buggy:", buggy_no_update("zjpc"), "| đúng:", min_time_to_type("zjpc"))

buggy: 27 | đúng: 34


Bản lỗi tính `a→z, a→j, a→p, a→c` = `1 + 9 + 11 + 2 + 4 = 27` thay vì `34`. Con trỏ **ở lại** chữ vừa gõ, không quay về `'a'`.

### Lỗi 4 — Dùng `26 - d` mà quên `min`, hoặc dùng `25 - d`

Có 26 ô nên tổng hai chiều là **26**, không phải 25. Thử nhanh: `a → z` ngược chiều đúng 1 bước = `26 - 25`.

In [22]:
d = abs(25 - 0)
print("26 - d =", 26 - d, "| 25 - d =", 25 - d, "(sai: a -> z mất 0 bước?)")

26 - d = 1 | 25 - d = 0 (sai: a -> z mất 0 bước?)


### Lỗi 5 — Dùng `%` kiểu C trong ngôn ngữ khác

Ở C++/Java, `(p - q) % 26` có thể **âm**. Nếu chuyển code sang ngôn ngữ đó, viết `((p - q) % 26 + 26) % 26`, hoặc dùng công thức `min(d, 26 - d)` với `d = abs(...)` (an toàn mọi ngôn ngữ).

## 14. Mẹo nhận dạng pattern

- Thấy "**vòng tròn**", "xoay vòng", "đồng hồ", "khoá số", "bánh xe" → chỉ số modulo `K` và khoảng cách `min(d, K - d)`.
- "Phải làm theo **thứ tự cố định**, mỗi bước có điểm đầu/cuối xác định" → các bước **độc lập** → tổng tối ưu = tổng tối ưu từng bước (tham lam đúng). Nếu có lựa chọn ảnh hưởng tới bước sau (ví dụ hai ngón tay — LeetCode 1320), tham lam **không** còn đúng mà cần DP.
- Cùng họ: 1629 (Slowest Key), 1165 (Single-Row Keyboard), 514 (Freedom Trail — vòng tròn + DP), 752 (Open the Lock — BFS trên khoá số).

## 15. Biến thể

### 15.1. Vòng tròn kích thước `K` bất kỳ (danh sách ký hiệu tuỳ ý)

In [23]:
def min_time_general(ring, word):
    pos = {ch: i for i, ch in enumerate(ring)}     # dict comprehension: ký hiệu -> vị trí
    K = len(ring)
    pointer, total = 0, 0
    for ch in word:
        t = pos[ch]
        d = abs(t - pointer)
        total += min(d, K - d) + 1
        pointer = t
    return total

print(min_time_general("abcdefghijklmnopqrstuvwxyz", "zjpc"))   # 34 — trùng bài gốc
print(min_time_general("0123456789", "907"))                    # khoá số 10 ô

34
8


### 15.2. LeetCode 1165 — Single-Row Keyboard (bàn phím **một hàng**, không vòng)

Không có chiều vòng qua → khoảng cách chỉ là `abs(d)`; và đề này không tính thời gian gõ.

In [24]:
def calculate_time(keyboard, word):
    pos = {ch: i for i, ch in enumerate(keyboard)}
    pointer, total = 0, 0
    for ch in word:
        total += abs(pos[ch] - pointer)
        pointer = pos[ch]
    return total

print(calculate_time("abcdefghijklmnopqrstuvwxyz", "cba"))   # 4

4


### 15.3. Khoá số nhiều vòng — tổng khoảng cách vòng tròn từng chữ số

Khoá có `m` bánh xe, mỗi bánh 10 chữ số. Số lần xoay ít nhất từ mã `a` sang mã `b` = tổng `min(d, 10 - d)` trên từng bánh (các bánh độc lập).

In [25]:
def lock_turns(a, b):
    return sum(min(abs(int(x) - int(y)), 10 - abs(int(x) - int(y))) for x, y in zip(a, b))

print(lock_turns("0000", "0202"), lock_turns("0000", "9999"), lock_turns("1234", "1234"))   # 4 4 0

4 4 0


## 16. Bài tập (TODO — tự làm, không chạy tự động)

**Bài 1 (dự đoán output):** `(3 - 5) % 26`, `abs(3 - 5)`, `min(20, 26 - 20)`, `chr(97 + 25)`.

**Bài 2 (tính tay):** lập bảng trace cho `word = "yes"` và cho kết quả.

**Bài 3 (điền chỗ trống):** hoàn thiện hàm khoảng cách vòng tròn.

In [26]:
def exercise_circular_distance(p, q, K=26):
    # TODO: d = |p - q|, trả về số nhỏ hơn giữa d và K - d
    pass

# assert exercise_circular_distance(0, 25) == 1
# assert exercise_circular_distance(9, 15) == 6
# assert exercise_circular_distance(4, 4) == 0

**Bài 4 (tự viết lại):** viết `exercise_min_time_to_type(word)` hoàn chỉnh (nhớ cộng thời gian gõ và cập nhật con trỏ).

In [27]:
def exercise_min_time_to_type(word):
    # TODO
    pass

# assert exercise_min_time_to_type("abc") == 5
# assert exercise_min_time_to_type("bza") == 7
# assert exercise_min_time_to_type("zjpc") == 34

**Bài 5 (biến thể):** viết `exercise_min_time_custom_start(word, start)` khi con trỏ ban đầu chỉ vào chữ `start` thay vì `'a'`.

**Bài 6 (mini challenge):** viết `exercise_best_start(word)` trả cặp `(chữ bắt đầu tốt nhất, thời gian)` — nếu được **chọn** vị trí ban đầu của con trỏ, chọn chữ nào để tổng thời gian nhỏ nhất? (Nếu nhiều chữ cùng tốt, lấy chữ nhỏ nhất theo bảng chữ.)

In [28]:
def exercise_min_time_custom_start(word, start):
    # TODO
    pass

def exercise_best_start(word):
    # TODO
    pass

# assert exercise_min_time_custom_start("abc", "a") == 5
# assert exercise_min_time_custom_start("zjpc", "z") == 33
# assert exercise_best_start("zjpc") == ("z", 33)

## 17. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | `%` của Python không âm | `-2 % 26` | `24, 2, 6, 'z'` |
| 2 | đổi chữ thành 0..25 | `y=24, e=4, s=18` | `a→y`: 2; `y→e`: 6; `e→s`: 12 |
| 3 | hai chiều | `d = abs(p - q)` | `return min(d, K - d)` |
| 4 | di chuyển + gõ | `total += min(d, 26 - d) + 1` | nhớ `pointer = target` |
| 5 | chỉ đổi giá trị ban đầu | `pointer = ord(start) - 97` | phần còn lại như bài 4 |
| 6 | thử cả 26 điểm bắt đầu | dùng hàm bài 5 | `min((time(c), c) for c in ...)` — so cặp theo thời gian trước |

## 18. Đáp án tham khảo — chỉ xem sau khi tự làm

**Bài 1:** `24`, `2`, `6`, `'z'`.

**Bài 2:** `a(0)→y(24)`: `d = 24` → 2 bước; `y(24)→e(4)`: `d = 20` → 6 bước; `e(4)→s(18)`: `d = 14` → 12 bước. Di chuyển `20`, gõ `3` → `23`.

In [29]:
print((3 - 5) % 26, abs(3 - 5), min(20, 26 - 20), chr(97 + 25))
print(min_time_to_type("yes"))


def answer_circular_distance(p, q, K=26):
    d = abs(p - q)
    return min(d, K - d)


def answer_min_time_to_type(word):
    pointer, total = 0, 0
    for ch in word:
        t = ord(ch) - ord("a")
        total += answer_circular_distance(pointer, t) + 1
        pointer = t
    return total


def answer_min_time_custom_start(word, start):
    pointer, total = ord(start) - ord("a"), 0
    for ch in word:
        t = ord(ch) - ord("a")
        total += answer_circular_distance(pointer, t) + 1
        pointer = t
    return total


def answer_best_start(word):
    best_time, best_ch = min((answer_min_time_custom_start(word, c), c) for c in "abcdefghijklmnopqrstuvwxyz")
    return best_ch, best_time


assert answer_circular_distance(0, 25) == 1
assert answer_circular_distance(9, 15) == 6
assert answer_circular_distance(4, 4) == 0
assert answer_min_time_to_type("abc") == 5
assert answer_min_time_to_type("bza") == 7
assert answer_min_time_to_type("zjpc") == 34
assert answer_min_time_custom_start("abc", "a") == 5
assert answer_min_time_custom_start("zjpc", "z") == 33
assert answer_best_start("zjpc") == ("z", 33)
for _ in range(200):
    w = "".join(random.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(random.randint(1, 30)))
    assert answer_min_time_to_type(w) == min_time_simulate(w)
    assert answer_min_time_custom_start(w, "a") == answer_min_time_to_type(w)
print("Đáp án tham khảo PASS toàn bộ test!")

24 2 6 z
23
Đáp án tham khảo PASS toàn bộ test!


## 19. Template nhớ nhanh + checklist submit

```python
class Solution:
    def minTimeToType(self, word: str) -> int:
        pointer = total = 0
        for ch in word:
            t = ord(ch) - ord("a")
            d = abs(t - pointer)
            total += min(d, 26 - d) + 1      # đường ngắn nhất trên vòng + 1 giây gõ
            pointer = t
        return total
```

**Câu thần chú:** *vòng 26 ô → khoảng cách `min(d, 26 − d)` → cộng 1 giây gõ → con trỏ ở lại chữ vừa gõ.*

**Checklist trước khi submit:**
- [ ] Con trỏ bắt đầu ở `'a'` (vị trí 0)?
- [ ] Xét cả chiều **vòng qua** `z|a` (`26 - d`)?
- [ ] Cộng **1 giây gõ** cho mỗi chữ?
- [ ] Cập nhật `pointer` sau mỗi chữ?
- [ ] Test `"z"` (2), `"n"` (14), `"aaaa"` (4)?